In [1]:
# Import Statments:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Adding Device Management:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("MPS is available and set as device.")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("CUDA is available and set as device.")
else:
    device = torch.device("cpu")
    print("Using CPU device.")


MPS is available and set as device.


In [2]:
peptides = []
amino_acids = []

max_len = 50

# Curating General Peptide Sequences from PeptideAtlas:
with open('APD_Hs_all.fasta', 'r', encoding='utf-8') as file:
    
    for line in file:
        if not line.startswith('>'):
            
            peptide = ['<SOS>']
            
            for c in range(len(line) - 1):
                peptide.append(line[c])
                amino_acids.append(line[c])
                
            peptide.append('<EOS>')
            
            for p in range(max_len - len(peptide)):
                peptide.append('<PAD>')
 
            if len(peptide) <= max_len:
                peptides.append(peptide)

# Tokenizing Amino Acids:
amino_acids.append('<SOS>')
amino_acids = sorted(list(set(amino_acids)))
amino_acids.append('<EOS>')
amino_acids.append('<PAD>')

vocab_size = len(amino_acids)

amino_acids


['<SOS>',
 'A',
 'C',
 'D',
 'E',
 'F',
 'G',
 'H',
 'I',
 'K',
 'L',
 'M',
 'N',
 'P',
 'Q',
 'R',
 'S',
 'T',
 'V',
 'W',
 'Y',
 '<EOS>',
 '<PAD>']

In [3]:
amps = []
non_amps = []

# Curating Antimicrobial Peptide Sequences from dbAMP:
with open('dbAMP3.fasta', 'r', encoding='utf-8') as file:
    for line in file:
        if not line.startswith('>'):
            
            amp = ['<SOS>']
            
            for c in range(len(line) - 1):
                if line[c] in amino_acids:
                    amp.append(line[c])
                
            amp.append('<EOS>')
            
            for p in range(max_len - len(amp)):
                amp.append('<PAD>')
 
            if len(amp) <= max_len:
                amps.append(amp)  

# Curating Non-Antimicrobial Peptide Sequences from UniProt:
with open('uniprotkb_non_amp.fasta', 'r', encoding='utf-8') as file:
    for line in file:
        if not line.startswith('>'):
            
            non_amp = ['<SOS>']
            
            for c in range(len(line) - 1):
                if line[c] in amino_acids:
                    non_amp.append(line[c])
                
            non_amp.append('<EOS>')
            
            for p in range(max_len - len(non_amp)):
                non_amp.append('<PAD>')
 
            if len(non_amp) <= max_len:
                non_amps.append(non_amp)  

non_amps = non_amps[:len(amps)]

len(peptides), len(amps), len(non_amps)


(318126, 28120, 28120)

In [4]:
# Creating Vocabularies:
itoaa = {aa:i for aa,i in enumerate(amino_acids)}
aatoi = {i:aa for aa,i in enumerate(amino_acids)}

encode = lambda l: [aatoi[aa] for aa in l]
decode = lambda l: ''.join([itoaa[i] for i in l])

# Creating Training/Validation Split:
n = int(0.9 * len(peptides))
m = int(0.9 * len(amps))

# General Peptides:
peptide_data = []

for peptide in peptides:
    peptide_data.append(encode(peptide))

peptide_data = torch.tensor(peptide_data, dtype=torch.long).to(device)

peptide_train_data = peptide_data[:n]
peptide_val_data = peptide_data[n:]

# AMP Tensors:
amp_data = []

for amp in amps:
    amp_data.append(encode(amp))
    
amp_data = torch.tensor(amp_data, dtype=torch.long).to(device)

amp_train_data = amp_data[:m]
amp_val_data = amp_data[m:]

# non-AMP Tensors:
non_amp_data = []

for non_amp in non_amps:
    non_amp_data.append(encode(non_amp))
    
non_amp_data = torch.tensor(non_amp_data, dtype=torch.long).to(device)

non_amp_train_data = non_amp_data[:m]
non_amp_val_data = non_amp_data[m:]

len(peptide_train_data), len(amp_train_data), len(non_amp_train_data)


(286313, 25308, 25308)

In [5]:
# Creating Hyperparameters:
n_embd = 128
head_size = 16
n_layer = 4
n_head = 4
batch_size = 32
block_size = 50
dropout = 0.1

# Single Head of Attention:
class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()

        # K,Q,V Matrices:
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        # Buffer Matrix and Dropout Layer:
        self.register_buffer('tril', torch.tril(torch.ones([block_size, block_size])))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        
        k = self.key(x)
        q = self.query(x)

        # Determining Affinities with Weighted Sum:
        wei = q @ k.transpose(-2, -1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        # Adjusting Embeddings With Value Matrix:
        v = self.value(x)
        out = wei @ v
        return out

# Parralelization of Attention Heads:
class MultiHeadedAttention(nn.Module):

    def __init__(self, head_size, n_head):
        super().__init__()

        # List of Attention Heads:
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_head)])

        # Projection and Dropout Layers:
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

# Multi-Layer Perceptron:
class FeedForward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()

        # Linear Layers, GELU and Dropout:
        self.net = nn.Sequential(
            nn.Linear(n_embd, n_embd * 4),
            nn.GELU(),
            nn.Linear(n_embd * 4, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

# Self-Attention/MLP Block:
class Block(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head

        # Self-Attention/MLP:
        self.sa = MultiHeadedAttention(head_size, n_head)
        self.ffwd = FeedForward(n_embd)

        # Layer Normalization:
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    # Residual Blocks and Self-Attention/MLP:
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# AMP Transformer Model:
class AMPTransformer(nn.Module):

    def __init__(self):
        super().__init__()

        # Token and Positional Embedding Tables:
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        # Block Layers:
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])

        # Layer Normalization and Unembedding:
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # Projection Head for Contrastive Learning:
        self.projection_head = nn.Sequential(
            nn.Linear(n_embd, n_embd),
            nn.GELU(),
            nn.Linear(n_embd, n_embd // 2),
        )

    # Create Embeddings for Contrastive Learning:
    def get_embeddings(self, idx):
        B,T = idx.shape

        # Embedding:
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb

        # Creating Logits after Forward Pass:
        x = self.blocks(x)
        x = self.ln_f(x)

        sequence_emb = torch.mean(x, dim=1)

        contrastive_emb = self.projection_head(sequence_emb)
        contrastive_emb = F.normalize(contrastive_emb, p=2, dim=1)
            
        return contrastive_emb

    def forward(self, idx, targets=None):
        B,T = idx.shape

        # Embedding:
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb

        # Creating Logits after Forward Pass:
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        # Determining Loss via Cross Entropy:
        if targets == None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.reshape(B*T, C)
            targets = targets.reshape(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx):

        # Generate New Data Until the <EOS> Token is Encountered:
        while True:
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_new = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, idx_new], dim=1)
            if idx_new == 21:
                break

        return idx

def contrastive_loss(embeddings_amp, embeddings_non_amp, margin=0.2):
    # Normalize embeddings
    embeddings_amp = F.normalize(embeddings_amp, p=2, dim=1)
    embeddings_non_amp = F.normalize(embeddings_non_amp, p=2, dim=1)
    
    # Positive pairs: AMP-AMP similarities (should be high)
    amp_sim_matrix = torch.matmul(embeddings_amp, embeddings_amp.T)
    
    # Get upper triangle excluding diagonal
    triu_indices = torch.triu_indices(amp_sim_matrix.size(0), amp_sim_matrix.size(1), offset=1)
    positive_sims = amp_sim_matrix[triu_indices[0], triu_indices[1]]
    
    # Negative pairs: AMP-nonAMP similarities (should be low)
    negative_sims = torch.matmul(embeddings_amp, embeddings_non_amp.T).flatten()
    
    # Margin loss
    positive_loss = torch.mean(torch.clamp(margin - positive_sims, min=0.0))
    negative_loss = torch.mean(torch.clamp(negative_sims + margin, min=0.0))
    
    return positive_loss + negative_loss
    
# Initializing Model:
model = AMPTransformer()
model = model.to(device)
model = torch.compile(model)

# Creating Optimizer:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

# Untrained Model Generation:
idx = torch.zeros([1, 1], dtype=torch.long).to(device)
print(decode(model.generate(idx)[0].tolist()))


<SOS>KPHRM<EOS>


In [11]:
from tqdm import tqdm
from torch.utils.data import DataLoader

# Creating Peptide Data Loader:
peptide_train_loader = DataLoader(peptide_train_data, batch_size=batch_size, shuffle=True)
peptide_val_loader = DataLoader(peptide_val_data, batch_size=batch_size, shuffle=False)        

# Creating AMP Data Loader:
amp_train_loader = DataLoader(amp_train_data, batch_size=batch_size, shuffle=True)
amp_val_loader = DataLoader(amp_val_data, batch_size=batch_size, shuffle=False)

# Creating non-AMP Data Loader:
non_amp_train_loader = DataLoader(non_amp_train_data, batch_size=batch_size, shuffle=True)
non_amp_val_loader = DataLoader(non_amp_val_data, batch_size=batch_size, shuffle=False)

# Creating Padding Tensor for Labels:
padding = []

for i in range(batch_size):
    padding.append(encode(['<PAD>']))

pad_tensor = torch.tensor(padding, dtype=torch.long).to(device)

# Pre-Training with General Peptides:
steps = 0

for step in range(steps):

    for batch, peptide in tqdm(enumerate(peptide_train_loader)):

        # Peptide Data [B, T+1]:
        peptide = torch.cat([peptide, pad_tensor], dim=1)

        # Indices [B, T]:
        x = peptide[:, :block_size]

        # Targets [B, T]:
        y = peptide[:, 1:block_size+1]

        logits, loss = model(x, y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        if batch == 8800:
            break

    print(f'step: {step} loss: {loss:.4f}')
        
print(f'total loss: {loss:.4f}')


total loss: 1.0104


In [12]:
# Trained Model Generation:
idx = torch.zeros([1, 1], dtype=torch.long).to(device)
print(decode(model.generate(idx)[0].tolist()))


<SOS>LNGLLPR<EOS>


In [14]:
# Contrastive Learning with AMP and non-AMP Data:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

steps = 0

for step in range(steps):

    for batch, (amp, non_amp) in tqdm(enumerate(zip(amp_train_loader, non_amp_train_loader))):
    
        # Creating Embeddings:
        amp_embeddings = model.get_embeddings(amp)
        non_amp_embeddings = model.get_embeddings(non_amp)
    
        # Calculate Contrastive Loss:
        loss = contrastive_loss(amp_embeddings, non_amp_embeddings, margin=0.2)
            
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    print(f'step: {step} loss: {loss:.4f}')

print(f'total loss: {loss:.4f}')


total loss: 0.1286


In [15]:
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader

# Curating MIC Data from GRAMPA:
url = "https://raw.githubusercontent.com/zswitten/Antimicrobial-Peptides/refs/heads/master/data/grampa.csv"
df = pd.read_csv(url)

# Curating Sequences:
MIC_sequences = df['sequence'].values

# Curating Values:
MIC_values = df['value'].values
MIC_values = np.power(10, MIC_values)
MIC_values = torch.tensor(np.log(MIC_values), dtype=torch.float32).to(device)

# Curating Sequence/Value Pairs:
MIC_data = []

for index, (sequence, value) in enumerate(zip(MIC_sequences, MIC_values)):

    tokenized_sequence = ['<SOS>']
            
    for c in range(len(sequence) - 1):
        if sequence[c] in amino_acids:
            tokenized_sequence.append(sequence[c])
                
    tokenized_sequence.append('<EOS>')
            
    for p in range(max_len - len(tokenized_sequence)):
        tokenized_sequence.append('<PAD>')
 
    if len(tokenized_sequence) <= max_len:
        MIC_data.append([torch.tensor(encode(tokenized_sequence), dtype=torch.long).to(device), value])

# Creating MIC Data Loader:
MIC_loader = DataLoader(MIC_data, batch_size=batch_size, shuffle=True) 

MIC_data[0]


[tensor([ 0,  6, 10, 13, 15,  9,  8, 10,  2,  1,  8,  1,  9,  9,  9,  6,  9,  2,
          9,  6, 13, 10,  9, 10, 18,  2,  9, 21, 22, 22, 22, 22, 22, 22, 22, 22,
         22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22],
        device='mps:0'),
 tensor(-0.9163, device='mps:0')]

In [63]:
# Creating Transformer with Regression Head:
class AMPTransformerMIC(nn.Module):
    def __init__(self, transformer_model, transformer_output_dim):
        super(AMPTransformerMIC, self).__init__()
        
        # Existing Transformer Mode:
        self.backbone = transformer_model
        
        # Regression Head for MIC Prediction:
        self.mic_regression_head = nn.Sequential(
            nn.Linear(transformer_output_dim, 128),
            nn.GELU(),
            nn.Linear(128, 1)
        )
            

    def forward(self, x):
        transformer_output, _ = self.backbone(x)
        pooled_output = transformer_output.mean(dim=1)
        mic_prediction = self.mic_regression_head(pooled_output)
        return mic_prediction

MIC_model = AMPTransformerMIC(model, vocab_size).to(device)

# Freezing Original Transformer Parameters:
for param in MIC_model.backbone.parameters():
    param.requires_grad = False

# Creating Optimizer:
optimizer = torch.optim.Adam(MIC_model.mic_regression_head.parameters(), lr=1e-3)

# Creating Criterion:
criterion = nn.MSELoss()

# Training Loop for MIC Prediction:
steps = 0

for step in range(steps):

    for batch, (sequence, MIC) in tqdm(enumerate(MIC_loader)):

        MIC = MIC.unsqueeze(dim=1)
        
        MIC_pred = MIC_model(sequence)
        
        loss = criterion(MIC_pred, MIC)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f'step: {step} loss: {loss:.4f}')


print(f'total loss: {loss:.4f}')


total loss: 0.6870


In [86]:
# Trained Model Generation:
idx = torch.zeros([1, 1], dtype=torch.long).to(device)
new_amp = model.generate(idx)[0].tolist()

amp_length = len(new_amp) if len(new_amp) < 50 else 49

for i in range(max_len - len(new_amp)):
    new_amp.append(22)

idx = torch.tensor([new_amp[:50]], dtype=torch.long).to(device)

sequence = ''.join(decode(new_amp[1:amp_length-1]))

print(f'Sequence {sequence}')
print(f'MIC {MIC_model(idx).item()}')


Sequence EIFQPITK
MIC -0.8138689994812012
